# Gradio with the OpenAI Agents SDK

## Learning goals

- Bridge Gradio's async chat callback to `Runner.run`
- Convert UI history into one deliberate SDK state strategy
- Render typed final output and a redacted event summary
- Bound turns and concurrency while handling SDK exceptions safely
- Keep live calls and server launch disabled during offline validation


## Keep four layers separate

1. **Gradio UI:** components, browser-session history, and presentation.
2. **UI adapter:** validates input and converts history/result shapes.
3. **Agents SDK runtime:** agent loop, tools, handoffs, guardrails, typed output, and traces.
4. **Provider/tool services:** network calls, credentials, data, and side effects.

The callback coordinates these layers; it should not mix HTML formatting, agent policy, authorization, and provider code into one large function.


## Mental model

```mermaid
sequenceDiagram
  participant B as Browser
  participant G as Gradio callback
  participant R as Runner
  participant A as Agent/tools/handoffs
  B->>G: New message plus UI history
  G->>G: Validate and build SDK input
  G->>R: await Runner.run(..., max_turns)
  R->>A: Execute bounded agent loop
  A-->>R: Typed final output and run items
  R-->>G: RunResult
  G->>G: Redact and map result
  G-->>B: Answer plus safe event summary
```

Gradio supports async callbacks, so call `await Runner.run(...)` directly. Avoid `Runner.run_sync` inside the async web request path.


## Define a UI-friendly final contract

Typed output makes rendering predictable. The UI still formats the object rather than dumping raw model or SDK structures.


In [ ]:
from pydantic import BaseModel, ConfigDict, Field


class TravelReply(BaseModel):
    model_config = ConfigDict(extra="forbid")
    answer: str = Field(min_length=1, max_length=1_500)
    assumptions: list[str] = Field(default_factory=list, max_length=6)


def render_reply(reply: TravelReply) -> str:
    if not reply.assumptions:
        return reply.answer
    assumptions = "\n".join(f"- {item}" for item in reply.assumptions)
    return f"{reply.answer}\n\nAssumptions:\n{assumptions}"


print(render_reply(TravelReply(answer="A relaxed Jaipur plan is possible.", assumptions=["No live pricing"])))


## Construct the agent once

Agent configuration is immutable enough to reuse for this stateless example. Per-user data and permissions must not be stored by mutating a shared agent; pass them through run context or authorized services.


In [ ]:
import os

from agents import Agent, ModelSettings

travel_chat_agent = Agent(
    name="Travel chat assistant",
    instructions=(
        "Answer practical travel-planning questions. State assumptions, do not claim live "
        "prices without a tool, and never claim to complete bookings or payments."
    ),
    model=os.getenv("OPENAI_MODEL", "gpt-4.1-mini"),
    model_settings=ModelSettings(max_tokens=450, store=False),
    output_type=TravelReply,
)
print(travel_chat_agent.name)


## Convert Gradio history deliberately

This notebook uses **client-managed history**: Gradio supplies prior display messages, and the adapter converts only string `user`/`assistant` entries into SDK input before appending the newest user turn.

Do not simultaneously pass this full history and `previous_response_id`, `conversation_id`, or a session containing the same turns. That can duplicate context. Browser history is also not authenticated memory; never use it to authorize tools or reveal private data.


In [ ]:
from typing import Any


def build_sdk_input(
    message: str,
    history: list[dict[str, Any]],
    *,
    max_history_messages: int = 12,
) -> list[dict[str, str]]:
    cleaned = (message or "").strip()
    if not cleaned:
        raise ValueError("Message cannot be blank.")
    if len(cleaned) > 2_000:
        raise ValueError("Message exceeds 2,000 characters.")

    converted = []
    for item in history[-max_history_messages:]:
        role = item.get("role")
        content = item.get("content")
        if role in {"user", "assistant"} and isinstance(content, str) and content.strip():
            converted.append({"role": role, "content": content})
    converted.append({"role": "user", "content": cleaned})
    return converted


sample_input = build_sdk_input(
    "What should I do on day two?",
    [
        {"role": "user", "content": "Plan Jaipur."},
        {"role": "assistant", "content": "What pace do you prefer?"},
    ],
)
print(sample_input)


## Map SDK items to safe UI events

Expose high-level item type and agent name. Do not return raw items, tool arguments, prompts, hidden reasoning, or provider responses in a public accordion.


In [ ]:
def safe_item_events(items: list[Any]) -> list[dict[str, str | None]]:
    events = []
    for item in items:
        agent = getattr(item, "agent", None)
        events.append({
            "type": type(item).__name__,
            "agent": getattr(agent, "name", None),
        })
    return events


print(safe_item_events([]))


## Async callback with bounded execution

The live flag keeps validation free of network calls. Specific SDK runtime failures become stable user messages; unexpected exceptions are logged server-side in a real application and returned as a generic error.


In [ ]:
from agents import (
    InputGuardrailTripwireTriggered,
    MaxTurnsExceeded,
    OutputGuardrailTripwireTriggered,
    RunConfig,
    Runner,
)

RUN_LIVE_CALL = False


async def chat(
    message: str,
    history: list[dict[str, Any]],
) -> tuple[str, list[dict[str, str | None]]]:
    try:
        sdk_input = build_sdk_input(message, history)
        if not RUN_LIVE_CALL:
            return (
                "Agents SDK UI is ready. Set RUN_LIVE_CALL = True to call the configured model.",
                [{"type": "offline_ready", "agent": travel_chat_agent.name}],
            )

        run_config = RunConfig(
            workflow_name="Gradio travel chat",
            trace_include_sensitive_data=False,
            trace_metadata={"interface": "gradio"},
        )
        result = await Runner.run(
            travel_chat_agent,
            sdk_input,
            max_turns=5,
            run_config=run_config,
        )
        reply: TravelReply = result.final_output
        return render_reply(reply), safe_item_events(result.new_items)
    except ValueError as error:
        return str(error), [{"type": "input_error", "agent": None}]
    except (InputGuardrailTripwireTriggered, OutputGuardrailTripwireTriggered):
        return "The request or response did not pass the configured policy.", [{"type": "guardrail", "agent": None}]
    except MaxTurnsExceeded:
        return "The assistant reached its step limit. Please narrow the request.", [{"type": "max_turns", "agent": None}]
    except Exception:
        return "Something went wrong. Please try again.", [{"type": "internal_error", "agent": None}]


offline_reply, offline_events = await chat("Plan Jaipur", [])
print(offline_reply)
print(offline_events)


## Build the Gradio UI

The JSON component shows allowlisted SDK events for the latest turn. `api_visibility="private"` avoids advertising the callback as a public API; deployment authentication and authorization remain separate controls.


In [ ]:
import gradio as gr


with gr.Blocks() as demo:
    gr.Markdown("# Agents SDK travel assistant")
    gr.Markdown("Live calls are disabled by default. Event details are intentionally redacted.")
    event_output = gr.JSON(label="Latest safe SDK events")
    gr.ChatInterface(
        fn=chat,
        chatbot=gr.Chatbot(label="Conversation", sanitize_html=True, height=420),
        additional_outputs=[event_output],
        examples=[
            "Plan two relaxed days in Jaipur.",
            "What assumptions affect a Jaipur budget?",
        ],
        api_visibility="private",
        concurrency_limit=4,
        save_history=False,
    )

print(type(demo).__name__)


## Optional launch

Launching starts a local server, so it is independently guarded from live model calls. Keep `share=False` for private course work.


In [ ]:
RUN_GRADIO = False

if RUN_GRADIO:
    demo.launch(
        inline=True,
        share=False,
        show_error=False,
    )
else:
    print("Gradio server not started.")


## Streaming and cancellation

For token or event streaming, use `Runner.run_streamed(...)` and consume `stream_events()` until it finishes. A streaming run is not complete until the iterator ends; summary properties and session persistence can still be settling. If a run is cancelled, continue consuming events so cleanup completes.

Start with non-streaming `Runner.run` until validation, error handling, and result mapping are correct. Streaming adds lifecycle complexity and should not expose raw event payloads to users.


## Production checklist

- Authenticate users before loading durable history or enabling private tools.
- Keep browser history separate from authorization and server identity.
- Bound message length, runner turns, tools, latency, concurrency, tokens, and cost.
- Avoid duplicate state strategies.
- Require approval and idempotency for side-effecting tools.
- Return safe errors; log protected diagnostics with correlation IDs.
- Disable sensitive trace payloads unless policy explicitly permits them.
- Test callback adapters with fake results before live/browser testing.
- Define history retention and deletion behavior.


## Exercise

1. Add one offline function tool and expose only its item type in UI events.
2. Replace client-managed history with one session strategy and remove history replay.
3. Add a fake `RunResult`-like object to test final-output and event mapping.
4. Add a timeout policy around the callback and decide what can be cancelled safely.
5. Enable streaming only after writing tests for completion, failure, and cancellation.


## Official references

- [Running agents](https://openai.github.io/openai-agents-python/running_agents/)
- [Streaming](https://openai.github.io/openai-agents-python/streaming/)
- [Run results](https://openai.github.io/openai-agents-python/results/)
- [Agents SDK tracing](https://openai.github.io/openai-agents-python/tracing/)


## Summary

A Gradio Agents SDK app is an async UI adapter around a bounded runner. Convert history once, keep state strategies distinct, render typed output, expose only redacted event summaries, catch SDK exceptions at the product boundary, and independently guard network calls and server launch during development.
